# RapidFire RAG Evaluation — 27 Pinecone + 9 BM25 Configs

Grid sweeps:
- **Chunk size** (namespace): `chunk256`, `chunk512`, `chunk1000` → 3 values  
- **Search type × k**: similarity, mmr, score_threshold@0.5 × k=5 → 3 values  
- **Reranker**: `ms-marco-MiniLM-L-4-v2`, `bge-reranker-base`, or none → 3 values  
- **Pinecone total: 3 × 3 × 3 = 27 configs | BM25 total: 3 × 1 × 3 = 9 configs**


In [1]:
import json
import re
import math
from pathlib import Path
from typing import List as listtype, Dict, Any

import pandas as pd
from datasets import Dataset

from rapidfireai import Experiment
from rapidfireai.automl import (
    List,
    RFLangChainRagSpec,
    RFOpenAIAPIModelConfig,
    RFPromptManager,
    RFGridSearch,
)
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

INFO 05-07 13:36:30 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 05-07 13:36:30 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.


## Config

In [4]:
TRITON_API_KEY   = Path("~/api-key.txt").expanduser().read_text(encoding="utf-8").splitlines()[0].strip()
PINECONE_API_KEY = Path("~/pinecone-api-key.txt").expanduser().read_text(encoding="utf-8").splitlines()[0].strip()

PINECONE_INDEX_NAME = "scifact"

NAMESPACE_CHUNK256  = "chunk256"
NAMESPACE_CHUNK512  = "chunk512"
NAMESPACE_CHUNK1000 = "chunk1000"

VALIDATION_PATH = Path("/home/sjrao/project1-cse234/project1_export/validation-set-golden-qa-pairs.json")

SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions about RapidFire AI's documentation. "
    "Answer thoroughly and cite specific details from the context. "
)

print("Config loaded.")

Config loaded.


## Helper Functions

In [5]:
import sys
import math
import json as _json

RETRIEVAL_K = 5
PROJECT1_METRICS_PATH = "/home/sjrao/project1-cse234/project1_export/Metrics"


def load_hf_dataset(path: Path) -> Dataset:
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    return Dataset.from_list([
        {
            "question_id":      item["question_id"],
            "question":         item["question"],
            "reference_answer": item["reference_answer"],
            "source_evidence":  json.dumps(item["source_evidence"]),  # JSON string to survive Arrow
        }
        for item in raw
    ])


def preprocess_fn(batch: Dict[str, listtype], rag, prompt_manager) -> Dict[str, listtype]:
    all_context = rag.get_context(batch_queries=batch["question"], serialize=False)
    serialized_context = rag.serialize_documents(all_context)

    retrieved_sources_json = []
    for docs in all_context:
        srcs = []
        for doc in docs:
            src = doc.metadata.get("source", "")
            line_start = doc.metadata.get("line_start", doc.metadata.get("start_line", 0))
            line_end   = doc.metadata.get("line_end",   doc.metadata.get("end_line",   0))
            if src:
                srcs.append({"file": src, "lines": [int(line_start), int(line_end)]})
        retrieved_sources_json.append(_json.dumps(srcs))  # JSON string — survives Arrow/Ray serialization

    prompts = [
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"Question:\n{question}\n\n"
                    f"Evidence:\n{context}\n\n"
                    "Answer:"
                ),
            },
        ]
        for question, context in zip(batch["question"], serialized_context)
    ]
    return {"prompts": prompts, "retrieved_context": serialized_context,
            "retrieved_sources_json": retrieved_sources_json, **batch}


def postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    return batch


def _tokenize(text: str) -> list:
    return re.findall(r"\w+", text.lower())


def _token_f1(pred: str, ref: str) -> float:
    p_tok = _tokenize(pred)
    r_tok = _tokenize(ref)
    if not p_tok or not r_tok:
        return 0.0
    common = set(p_tok) & set(r_tok)
    if not common:
        return 0.0
    precision = len(common) / len(p_tok)
    recall    = len(common) / len(r_tok)
    return 2 * precision * recall / (precision + recall)


def _overlap(a, b):
    return a[0] == b[0] and a[1] <= b[2] and b[1] <= a[2]


def _ndcg_at_k(retrieved_spans, gt_spans, k):
    top = retrieved_spans[:k]
    relevance = [1 if any(_overlap(r, g) for g in gt_spans) else 0 for r in top]
    dcg  = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevance))
    ideal = sorted(relevance, reverse=True)
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def _mrr(retrieved_spans, gt_spans):
    for i, span in enumerate(retrieved_spans):
        if any(_overlap(span, g) for g in gt_spans):
            return 1.0 / (i + 1)
    return 0.0


def compute_metrics_fn(batch: Dict[str, listtype]) -> Dict[str, Dict[str, Any]]:
    # Import inside fn so Ray workers can find the module
    import sys, math, json as _json
    if PROJECT1_METRICS_PATH not in sys.path:
        sys.path.insert(0, PROJECT1_METRICS_PATH)
    from project1_eval import f1_at_k, precision_at_k, recall_at_k, to_spans

    def _norm(items):
        """Normalize to plain Python dicts for to_spans (handles Arrow struct types)."""
        out = []
        for x in (items or []):
            try:
                out.append({"file": str(x["file"]),
                            "lines": [int(x["lines"][0]), int(x["lines"][1])]})
            except Exception:
                pass
        return out

    total = len(batch["question"])
    k = RETRIEVAL_K

    token_f1s, precisions, recalls, ret_f1s, ndcgs, mrrs = [], [], [], [], [], []

    for pred, ref, ret_srcs_json, gt_srcs_json in zip(
        batch["generated_text"],
        batch["reference_answer"],
        batch["retrieved_sources_json"],
        batch["source_evidence"],
    ):
        token_f1s.append(_token_f1(pred, ref))

        try:
            ret_srcs = _json.loads(ret_srcs_json) if ret_srcs_json else []
            gt_srcs  = _json.loads(gt_srcs_json)  if gt_srcs_json  else []
            gt_spans  = to_spans(_norm(gt_srcs))
            ret_spans = to_spans(_norm(ret_srcs))
        except Exception:
            gt_spans = ret_spans = []

        precisions.append(precision_at_k(ret_spans, gt_spans, k))
        recalls.append(recall_at_k(ret_spans, gt_spans, k))
        ret_f1s.append(f1_at_k(ret_spans, gt_spans, k))
        ndcgs.append(_ndcg_at_k(ret_spans, gt_spans, k))
        mrrs.append(_mrr(ret_spans, gt_spans))

    return {
        "Total":              {"value": total},
        "Token F1":           {"value": sum(token_f1s)  / total},
        f"Precision@{k}":     {"value": sum(precisions) / total},
        f"Recall@{k}":        {"value": sum(recalls)    / total},
        f"F1@{k}":            {"value": sum(ret_f1s)    / total},
        f"NDCG@{k}":          {"value": sum(ndcgs)      / total},
        "MRR":                {"value": sum(mrrs)       / total},
    }


def accumulate_metrics_fn(aggregated: Dict[str, listtype]) -> Dict[str, Dict[str, Any]]:
    counts = [m["value"] for m in aggregated["Total"]]
    total  = sum(counts)
    algebraic = ["Token F1", f"Precision@{RETRIEVAL_K}", f"Recall@{RETRIEVAL_K}",
                 f"F1@{RETRIEVAL_K}", f"NDCG@{RETRIEVAL_K}", "MRR"]
    return {
        "Total": {"value": total},
        **{
            metric: {
                "value": sum(m["value"] * n for m, n in zip(aggregated[metric], counts)) / total,
                "is_algebraic": True,
                "value_range":  (0, 1),
            }
            for metric in algebraic
        },
    }


def resolve_index_name(api_key: str, preferred: str) -> str:
    from pinecone import Pinecone
    pc        = Pinecone(api_key=api_key)
    available = [idx.name for idx in pc.list_indexes()]
    print(f"Available Pinecone indexes: {available}")
    if preferred in available:
        return preferred
    if available:
        print(f"Index '{preferred}' not found — using '{available[0]}' instead.")
        return available[0]
    raise RuntimeError("No Pinecone indexes found. Check PINECONE_API_KEY.")


print("Functions defined.")


Functions defined.


## Load Dataset

In [6]:
dataset = load_hf_dataset(VALIDATION_PATH)
print(f"Loaded {len(dataset)} questions.")
dataset.to_pandas().head()

Loaded 45 questions.


,question_id,question,reference_answer,source_evidence
0,1,What are the two knob set generators currently...,RapidFire AI currently supports two knob set g...,"[{""file"": ""configs.rst"", ""lines"": [23, 48]}]"
1,2,How does RapidFire AI's adaptive execution eng...,Traditional tools force you to run one config ...,"[{""file"": ""difference.rst"", ""lines"": [4, 29]}]"
2,3,What parameters does the Experiment constructo...,The Experiment constructor accepts three param...,"[{""file"": ""experiment.rst"", ""lines"": [27, 39]}]"
3,4,What is the difference between RFGridSearch an...,RFGridSearch requires each knob to have either...,"[{""file"": ""configs.rst"", ""lines"": [58, 102]}]"
4,5,How do I set up RapidFire AI for RAG evaluatio...,For RAG/context engineering with only closed m...,"[{""file"": ""walkthroughrag.rst"", ""lines"": [29, ..."


In [ ]:
# from langchain_pinecone import PineconeVectorStore
# from langchain_huggingface import HuggingFaceEmbeddings as _HFE

# index_name = resolve_index_name(PINECONE_API_KEY, PINECONE_INDEX_NAME)

# emb = _HFE(
#     model_name="BAAI/bge-small-en-v1.5",
#     model_kwargs={"device": "cpu"},
#     encode_kwargs={"normalize_embeddings": True},
# )
# vs = PineconeVectorStore(
#     index_name=index_name,
#     embedding=emb,
#     namespace=NAMESPACE_CHUNK512,
#     pinecone_api_key=PINECONE_API_KEY,
# )
# docs = vs.similarity_search("what is an experiment", k=2)
# for i, d in enumerate(docs):
#     print(f"--- Doc {i} metadata ---")
#     for k, v in d.metadata.items():
#         print(f"  {k!r}: {v!r}")


Available Pinecone indexes: ['project1-rag-index', 'langchain2', 'pinecone']
Index 'scifact' not found — using 'project1-rag-index' instead.
--- Doc 0 metadata ---
  'chunk_index': 4.0
  'line_end': 62.0
  'line_start': 50.0
  'source': 'glossary.rst'
  'start_index': 1795.0
--- Doc 1 metadata ---
  'chunk_index': 0.0
  'line_end': 10.0
  'line_start': 1.0
  'source': 'experiment.rst'
  'start_index': 0.0


## Build RAG + Model Config (48 configs)

In [8]:
index_name = resolve_index_name(PINECONE_API_KEY, PINECONE_INDEX_NAME)

embedding_cfg = {
    "class": HuggingFaceEmbeddings,
    "model_name": "BAAI/bge-small-en-v1.5",
    "model_kwargs": {"device": "cpu"},
    "encode_kwargs": {"normalize_embeddings": True},
}

RERANKER_CONFIGS = [
    {
        "class":        CrossEncoderReranker,
        "model_name":   "cross-encoder/ms-marco-MiniLM-L-4-v2",
        "model_kwargs": {"device": "cpu"},
        "top_n":        5,
    },
    {
        "class":        CrossEncoderReranker,
        "model_name":   "BAAI/bge-reranker-base",
        "model_kwargs": {"device": "cpu"},
        "top_n":        5,
    },
    None,
]

rag_spec = RFLangChainRagSpec(
    vector_store_cfg={
        "type": "pinecone",
        "pinecone_api_key": PINECONE_API_KEY,
        "index_namespace": List([
            (index_name, NAMESPACE_CHUNK256),
            (index_name, NAMESPACE_CHUNK512),
            (index_name, NAMESPACE_CHUNK1000),
        ]),
        "embedding_cfg": embedding_cfg,
    },
    search_cfg=List([
        {"type": "similarity",                 "k": 5},
        {"type": "mmr",                        "k": 5},
        {"type": "similarity_score_threshold", "k": 5, "score_threshold": 0.5},
    ]),
    reranker_cfg=List(RERANKER_CONFIGS),
    enable_gpu_search=False,
)

openai_cfg = RFOpenAIAPIModelConfig(
    client_config={
        "api_key":     TRITON_API_KEY,
        "base_url":    "https://tritonai-api.ucsd.edu",
        "max_retries": 2,
    },
    model_config={
        "model":                "api-mistral-small-3.2-2506",
        "max_completion_tokens": 1024,
    },
    rpm_limit=120,
    tpm_limit=1_000_000,
    rag=rag_spec,
    prompt_manager=None,
)

config_set = List([{
    "openai_config":          openai_cfg,
    "batch_size":             9,
    "preprocess_fn":          preprocess_fn,
    "postprocess_fn":         postprocess_fn,
    "compute_metrics_fn":     compute_metrics_fn,
    "accumulate_metrics_fn":  accumulate_metrics_fn,
    "online_strategy_kwargs": {
        "strategy_name":    "normal",
        "confidence_level": 0.95,
        "use_fpc":          True,
    },
}])

config_group = RFGridSearch(config_set)
print("Config group built.")


Available Pinecone indexes: ['project1-rag-index', 'langchain2', 'pinecone']
Index 'scifact' not found — using 'project1-rag-index' instead.
Config group built.


## Build BM25 Retrievers (in-memory)

In [ ]:
import bisect
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever

SOURCEDOCS_PATH  = Path("/home/sjrao/project1-cse234/project1_export/sourcedocs")
BM25_CHUNK_SIZES = [256, 512, 1000]
BM25_K_VALUES    = [5]

raw_docs = []
file_line_offsets = {}  
for rst_file in sorted(SOURCEDOCS_PATH.glob("*.rst")):
    text = rst_file.read_text(encoding="utf-8", errors="replace")
    raw_docs.append(Document(page_content=text, metadata={"source": rst_file.name}))
    offsets = [0]
    for i, c in enumerate(text):
        if c == "\n":
            offsets.append(i + 1)
    file_line_offsets[rst_file.name] = offsets
print(f"Loaded {len(raw_docs)} RST files")

def _char_to_line(offsets, char_idx):
    """Return 1-indexed line number for the given character offset."""
    return bisect.bisect_right(offsets, char_idx)

retriever_map = {}
for chunk_size in BM25_CHUNK_SIZES:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=max(20, chunk_size // 10),
        add_start_index=True,  # adds start_index (char offset) to each chunk's metadata
    )
    chunks = splitter.split_documents(raw_docs)
    # Compute line_start / line_end from character offsets
    for chunk in chunks:
        src       = chunk.metadata.get("source", "")
        start_idx = chunk.metadata.get("start_index", 0)
        end_idx   = start_idx + len(chunk.page_content)
        if src in file_line_offsets:
            offs = file_line_offsets[src]
            chunk.metadata["line_start"] = _char_to_line(offs, start_idx)
            chunk.metadata["line_end"]   = _char_to_line(offs, end_idx)
    print(f"  chunk_size={chunk_size} → {len(chunks)} chunks")
    for k in BM25_K_VALUES:
        retriever_map[(chunk_size, k)] = BM25Retriever.from_documents(chunks, k=k)

print("BM25 retrievers ready.")


Loaded 31 RST files
  chunk_size=256 → 1068 chunks
  chunk_size=512 → 497 chunks
  chunk_size=1000 → 236 chunks
BM25 retrievers ready.


## Run BM25 Sweep (9 configs = 3 chunk sizes × 1 k value × 3 rerankers)


In [10]:
bm25_configs = []
for chunk_size in BM25_CHUNK_SIZES:
    for k in BM25_K_VALUES:
        retriever = retriever_map[(chunk_size, k)]
        for reranker in RERANKER_CONFIGS:
            bm25_rag = RFLangChainRagSpec(retriever=retriever, reranker_cfg=reranker)
            bm25_openai_cfg = RFOpenAIAPIModelConfig(
                client_config={
                    "api_key":     TRITON_API_KEY,
                    "base_url":    "https://tritonai-api.ucsd.edu",
                    "max_retries": 2,
                },
                model_config={
                    "model":                 "api-mistral-small-3.2-2506",
                    "max_completion_tokens": 1024,
                },
                rpm_limit=120,
                tpm_limit=1_000_000,
                rag=bm25_rag,
                prompt_manager=None,
            )
            bm25_configs.append({
                "openai_config":          bm25_openai_cfg,
                "batch_size":             9,
                "preprocess_fn":          preprocess_fn,
                "postprocess_fn":         postprocess_fn,
                "compute_metrics_fn":     compute_metrics_fn,
                "accumulate_metrics_fn":  accumulate_metrics_fn,
                "online_strategy_kwargs": {
                    "strategy_name":    "normal",
                    "confidence_level": 0.95,
                    "use_fpc":          True,
                },
            })

print(f"Built {len(bm25_configs)} BM25 configs.")


Built 9 BM25 configs.


In [11]:
bm25_experiment = Experiment(
    experiment_name="bm25-grid-9configs-validation",
    mode="evals",
)

print("Launching BM25 run_evals — 9 configs, 45 questions each ...")
bm25_results = bm25_experiment.run_evals(
    config_group=RFGridSearch(List(bm25_configs)),
    dataset=dataset,
    num_shards=5,
    seed=42,
)
bm25_experiment.end()

The previously running experiment bm25-grid-9configs-validation_4 was forcibly ended. Created a new experiment 'bm25-grid-9configs-validation_5' with Experiment ID: 30 at /home/sjrao/rapidfireai/rapidfire_experiments/bm25-grid-9configs-validation_5
Launching BM25 run_evals — 9 configs, 45 questions each ...


=== Preprocessing RAG Sources ===


RAG Source ID,Status,Duration,Device
1,Complete,25.0s,CPU



=== Multi-Config Experiment Progress ===


Run ID,Model,Status,Progress,Conf. Interval,search_cfg.k,search_cfg.type,reranker_cfg.class,reranker_cfg.model_name,reranker_cfg.top_n,model_config,F1@5,MRR,NDCG@5,Precision@5,Processing Time,Recall@5,Samples Per Second,Samples Processed,Throughput,Token F1,Total
1,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,5,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L-4-v2,5,max_completion_tokens=1024,"0.3291 [0.3291, 0.3291]","61.48% [61.48%, 61.48%]","62.32% [62.32%, 62.32%]","0.2311 [0.2311, 0.2311]",462.79 seconds,"0.7000 [0.7000, 0.7000]",0.10,45,0.1/s,"0.1897 [0.1897, 0.1897]",45
2,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,5,similarity,CrossEncoderReranker,BAAI/bge-reranker-base,5,max_completion_tokens=1024,"0.3291 [0.3291, 0.3291]","61.93% [61.93%, 61.93%]","63.60% [63.60%, 63.60%]","0.2311 [0.2311, 0.2311]",434.04 seconds,"0.7000 [0.7000, 0.7000]",0.10,45,0.1/s,"0.1852 [0.1852, 0.1852]",45
3,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,5,similarity,-,-,-,max_completion_tokens=1024,"0.3291 [0.3291, 0.3291]","47.74% [47.74%, 47.74%]","53.38% [53.38%, 53.38%]","0.2311 [0.2311, 0.2311]",424.21 seconds,"0.7000 [0.7000, 0.7000]",0.11,45,0.1/s,"0.1856 [0.1856, 0.1856]",45
4,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,5,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L-4-v2,5,max_completion_tokens=1024,"0.3291 [0.3291, 0.3291]","61.48% [61.48%, 61.48%]","62.32% [62.32%, 62.32%]","0.2311 [0.2311, 0.2311]",417.10 seconds,"0.7000 [0.7000, 0.7000]",0.11,45,0.1/s,"0.1844 [0.1844, 0.1844]",45
5,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,5,similarity,CrossEncoderReranker,BAAI/bge-reranker-base,5,max_completion_tokens=1024,"0.3291 [0.3291, 0.3291]","61.93% [61.93%, 61.93%]","63.60% [63.60%, 63.60%]","0.2311 [0.2311, 0.2311]",406.90 seconds,"0.7000 [0.7000, 0.7000]",0.11,45,0.1/s,"0.1887 [0.1887, 0.1887]",45
6,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,5,similarity,-,-,-,max_completion_tokens=1024,"0.3291 [0.3291, 0.3291]","47.74% [47.74%, 47.74%]","53.38% [53.38%, 53.38%]","0.2311 [0.2311, 0.2311]",395.58 seconds,"0.7000 [0.7000, 0.7000]",0.11,45,0.1/s,"0.1856 [0.1856, 0.1856]",45
7,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,5,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L-4-v2,5,max_completion_tokens=1024,"0.3291 [0.3291, 0.3291]","61.48% [61.48%, 61.48%]","62.32% [62.32%, 62.32%]","0.2311 [0.2311, 0.2311]",390.30 seconds,"0.7000 [0.7000, 0.7000]",0.12,45,0.1/s,"0.1835 [0.1835, 0.1835]",45
8,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,5,similarity,CrossEncoderReranker,BAAI/bge-reranker-base,5,max_completion_tokens=1024,"0.3291 [0.3291, 0.3291]","61.93% [61.93%, 61.93%]","63.60% [63.60%, 63.60%]","0.2311 [0.2311, 0.2311]",380.71 seconds,"0.7000 [0.7000, 0.7000]",0.12,45,0.1/s,"0.1451 [0.1451, 0.1451]",45
9,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,5,similarity,-,-,-,max_completion_tokens=1024,"0.3291 [0.3291, 0.3291]","47.74% [47.74%, 47.74%]","53.38% [53.38%, 53.38%]","0.2311 [0.2311, 0.2311]",369.36 seconds,"0.7000 [0.7000, 0.7000]",0.12,45,0.1/s,"0.1972 [0.1972, 0.1972]",45


Experiment bm25-grid-9configs-validation_5 ended


## Run Experiment

In [12]:
experiment = Experiment(
    experiment_name="rag-grid-27configs-validation",
    mode="evals",
)

print("Launching run_evals — 27 configs, 45 questions each ...")
results = experiment.run_evals(
    config_group=config_group,
    dataset=dataset,
    num_shards=5,
    seed=42,
)


An experiment with the same name already exists. Created a new experiment 'rag-grid-27configs-validation_1' with Experiment ID: 31 at /home/sjrao/rapidfireai/rapidfire_experiments/rag-grid-27configs-validation_1
Launching run_evals — 27 configs, 45 questions each ...


=== Preprocessing RAG Sources ===


RAG Source ID,Status,Duration,Device,Vector Store,Index Name,Namespace
1,Complete,26.0s,CPU,Pinecone,project1-rag-index,chunk256
2,Complete,26.0s,CPU,Pinecone,project1-rag-index,chunk512
3,Complete,26.5s,CPU,Pinecone,project1-rag-index,chunk1000



=== Multi-Config Experiment Progress ===


Run ID,Model,Status,Progress,Conf. Interval,embedding_cfg.class,embedding_cfg.encode_kwargs.normalize_embeddings,embedding_cfg.model_name,vector_store_cfg.index_namespace,vector_store_cfg.type,search_cfg.fetch_k,search_cfg.k,search_cfg.lambda_mult,search_cfg.score_threshold,search_cfg.type,reranker_cfg.class,reranker_cfg.model_name,reranker_cfg.top_n,model_config,F1@5,MRR,NDCG@5,Precision@5,Processing Time,Recall@5,Samples Per Second,Samples Processed,Throughput,Token F1,Total
1,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,HuggingFaceEmbeddings,True,BAAI/bge-small-en-v1.5,"('project1-rag-index', 'chunk256')",pinecone,-,5,-,-,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L-4-v2,5,max_completion_tokens=1024,"0.3952 [0.3952, 0.3952]","67.26% [67.26%, 67.26%]","68.76% [68.76%, 68.76%]","0.2856 [0.2856, 0.2856]",2126.07 seconds,"0.7500 [0.7500, 0.7500]",0.02,45,0.0/s,"0.2133 [0.2133, 0.2133]",45
2,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,HuggingFaceEmbeddings,True,BAAI/bge-small-en-v1.5,"('project1-rag-index', 'chunk256')",pinecone,-,5,-,-,similarity,CrossEncoderReranker,BAAI/bge-reranker-base,5,max_completion_tokens=1024,"0.3952 [0.3952, 0.3952]","69.52% [69.52%, 69.52%]","70.34% [70.34%, 70.34%]","0.2856 [0.2856, 0.2856]",2092.51 seconds,"0.7500 [0.7500, 0.7500]",0.02,45,0.0/s,"0.2054 [0.2054, 0.2054]",45
3,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,HuggingFaceEmbeddings,True,BAAI/bge-small-en-v1.5,"('project1-rag-index', 'chunk256')",pinecone,-,5,-,-,similarity,-,-,-,max_completion_tokens=1024,"0.3952 [0.3952, 0.3952]","58.07% [58.07%, 58.07%]","62.78% [62.78%, 62.78%]","0.2856 [0.2856, 0.2856]",2077.51 seconds,"0.7500 [0.7500, 0.7500]",0.02,45,0.0/s,"0.2126 [0.2126, 0.2126]",45
4,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,HuggingFaceEmbeddings,True,BAAI/bge-small-en-v1.5,"('project1-rag-index', 'chunk256')",pinecone,20,5,0.5,-,mmr,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L-4-v2,5,max_completion_tokens=1024,"0.3672 [0.3672, 0.3672]","67.59% [67.59%, 67.59%]","70.13% [70.13%, 70.13%]","0.2489 [0.2489, 0.2489]",2068.50 seconds,"0.7722 [0.7722, 0.7722]",0.02,45,0.0/s,"0.2159 [0.2159, 0.2159]",45
5,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,HuggingFaceEmbeddings,True,BAAI/bge-small-en-v1.5,"('project1-rag-index', 'chunk256')",pinecone,20,5,0.5,-,mmr,CrossEncoderReranker,BAAI/bge-reranker-base,5,max_completion_tokens=1024,"0.3672 [0.3672, 0.3672]","68.89% [68.89%, 68.89%]","70.80% [70.80%, 70.80%]","0.2489 [0.2489, 0.2489]",2056.11 seconds,"0.7722 [0.7722, 0.7722]",0.02,45,0.0/s,"0.2175 [0.2175, 0.2175]",45
6,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,HuggingFaceEmbeddings,True,BAAI/bge-small-en-v1.5,"('project1-rag-index', 'chunk256')",pinecone,20,5,0.5,-,mmr,-,-,-,max_completion_tokens=1024,"0.3672 [0.3672, 0.3672]","59.59% [59.59%, 59.59%]","63.15% [63.15%, 63.15%]","0.2489 [0.2489, 0.2489]",2040.90 seconds,"0.7722 [0.7722, 0.7722]",0.02,45,0.0/s,"0.2047 [0.2047, 0.2047]",45
7,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,HuggingFaceEmbeddings,True,BAAI/bge-small-en-v1.5,"('project1-rag-index', 'chunk256')",pinecone,-,5,-,0.5,similarity_score_threshold,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L-4-v2,5,max_completion_tokens=1024,"0.3952 [0.3952, 0.3952]","67.26% [67.26%, 67.26%]","68.76% [68.76%, 68.76%]","0.2856 [0.2856, 0.2856]",2029.92 seconds,"0.7500 [0.7500, 0.7500]",0.02,45,0.0/s,"0.2078 [0.2078, 0.2078]",45
8,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,HuggingFaceEmbeddings,True,BAAI/bge-small-en-v1.5,"('project1-rag-index', 'chunk256')",pinecone,-,5,-,0.5,similarity_score_threshold,CrossEncoderReranker,BAAI/bge-reranker-base,5,max_completion_tokens=1024,"0.3952 [0.3952, 0.3952]","69.52% [69.52%, 69.52%]","70.34% [70.34%, 70.34%]","0.2856 [0.2856, 0.2856]",2018.41 seconds,"0.7500 [0.7500, 0.7500]",0.02,45,0.0/s,"0.2102 [0.2102, 0.2102]",45
9,api-mistral-small-3.2-2506,COMPLETED,5/5,0.000,HuggingFaceEmbeddings,True,BAAI/bge-small-en-v1.5,"('project1-rag-index', '

## Results

In [13]:
def results_to_df(results, retriever_type):
    rows = []
    for run_id, (_, metrics) in results.items():
        row = {k: v["value"] if isinstance(v, dict) and "value" in v else v for k, v in metrics.items()}
        row["run_id"] = run_id
        row["retriever_type"] = retriever_type
        rows.append(row)
    return pd.DataFrame(rows)

df_pinecone = results_to_df(results, "pinecone")
df_bm25     = results_to_df(bm25_results, "bm25")
df_all      = pd.concat([df_pinecone, df_bm25], ignore_index=True)

df_all.sort_values("Token F1", ascending=False)

,run_id,model_name,embedding_cfg,vector_store_cfg,search_cfg,reranker_cfg,model_config,Samples Processed,Processing Time,Samples Per Second,Total,Token F1,Precision@5,Recall@5,F1@5,NDCG@5,MRR,retriever_type
20,21,api-mistral-small-3.2-2506,"{'class': 'HuggingFaceEmbeddings', 'model_name...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'similarity', 'k': 5}",NaN,{'max_completion_tokens': 1024},45,1834.24 seconds,0.02,45,0.285025,0.293333,0.883333,0.425397,0.812635,0.795185,pinecone
19,20,api-mistral-small-3.2-2506,"{'class': 'HuggingFaceEmbeddings', 'model_name...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'similarity', 'k': 5}","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 1024},45,1857.29 seconds,0.02,45,0.283973,0.293333,0.883333,0.425397,0.869895,0.872222,pinecone
26,27,api-mistral-small-3.2-2506,"{'class': 'HuggingFaceEmbeddings', 'model_name...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'similarity_score_threshold', 'k': 5,...",NaN,{'max_completion_tokens': 1024},45,1715.17 seconds,0.03,45,0.283874,0.293333,0.883333,0.425397,0.812635,0.795185,pinecone
25,26,api-mistral-small-3.2-2506,"{'class': 'HuggingFaceEmbeddings', 'model_name...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'similarity_score_threshold', 'k': 5,...","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 1024},45,1735.54 seconds,0.03,45,0.283232,0.293333,0.883333,0.425397,0.869895,0.872222,pinecone
18,19,api-mistral-small-3.2-2506,"{'class': 'HuggingFaceEmbeddings', 'model_name...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'similarity', 'k': 5}","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 1024},45,1872.42 seconds,0.02,45,0.279194,0.293333,0.883333,0.425397,0.880121,0.904444,pinecone
22,23,api-mistral-small-3.2-2506,"{'class': 'HuggingFaceEmbeddings', 'model_name...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'mmr', 'k': 5, 'fetch_k': 20, 'lambda...","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 1024},45,1792.06 seconds,0.03,45,0.272806,0.217778,0.850000,0.339330,0.893947,0.894444,pinecone
21,22,api-mistral-small-3.2-2506,"{'class': 'HuggingFaceEmbeddings', 'model_name...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'mmr', 'k': 5, 'fetch_k': 20, 'lambda...","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 1024},45,1816.31 seconds,0.02,45,0.270815,0.217778,0.850000,0.339330,0.885563,0.882222,pinecone
23,24,api-mistral-small-3.2-2506,"{'class': 'HuggingFaceEmbeddings', 'model_name...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'mmr', 'k': 5, 'fetch_k': 20, 'lambda...",NaN,{'max_completion_tokens': 1024},45,1764.56 seconds,0.03,45,0.264499,0.217778,0.850000,0.339330,0.779776,0.757037,pinecone
9,10,api-mistral-small-3.2-2506,"{'class': 'HuggingFaceEmbeddings', 'model_name...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'similarity', 'k': 5}","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 1024},45,1992.52 seconds,0.02,45,0.244228,0.293333,0.888889,0.418911,0.828983,0.801852,pinecone
17,18,api-mistral-small-3.2-2506,"{'class': 'HuggingFaceEmbeddings', 'model_name...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'similarity_score_threshold', 'k': 5,...",NaN,{'max_completion_tokens': 1024},45,1882.06 seconds,0.02,45,0.240256,0.293333,0.888889,0.418911,0.731120,0.661852,pinecone


In [14]:
out_csv = Path("all_eval_results.csv")
df_all.to_csv(out_csv, index=False)
print(f"Saved → {out_csv}")

log_pinecone = experiment.get_log_file_path()
log_bm25     = bm25_experiment.get_log_file_path()
print(f"Pinecone log → {log_pinecone}")
print(f"BM25 log     → {log_bm25}")

Saved → all_eval_results.csv
Pinecone log → /home/sjrao/rapidfireai/logs/rag-grid-27configs-validation_1/rapidfire.log
BM25 log     → /home/sjrao/rapidfireai/logs/bm25-grid-9configs-validation_5/rapidfire.log


In [15]:
out_csv = Path("all_eval_results.csv")
df_all.to_csv(out_csv, index=False)